In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setting display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported and display options set.")

In [ ]:
# Loading the datasets
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
product_category_name_translation = pd.read_csv('../data/product_category_name_translation.csv')
geolocation = pd.read_csv('../data/olist_geolocation_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')


print(f"Orders: {orders.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Products: {products.shape}")
print(f"Customers: {customers.shape}")
print(f"Sellers: {sellers.shape}")
print(f"Reviews: {reviews.shape}")

print("\n geolocation")
print(geolocation.info())
print("\n payments")
print(payments.info())

In [ ]:
# Quick look at the first few rows of each dataset
print("**** Orders Sample ****")
print(orders.head())
print("\n**** Order Info ****")
print(orders.info())
print("\n**** Orders Description ****")
print(orders.describe())

# Checking for missing values in orders dataset
print("\n**** Orders Missing Values ****")
print(orders.isnull().sum())

# Looking at the order statuses
print("\n**** Order Status Distribution ****")
print(orders['order_status'].value_counts())

In [ ]:
# This is where the pricing data is
print("\n**** Order Items (Pricing Data) ****")
print(order_items.head())
print("\n")
print(order_items.info())

# Price statistics
print("\n**** Price Statistics ****")
print(order_items['price'].describe())

# On average the price is 120 BRL, but there are some very expensive products (max price is 6735 BRL). The standard deviation is quite high (183.63 BRL), 
# which indicates a wide range of prices in the dataset. The minimum price is 0.85 BRL, which could indicate free products or data entry errors. 
# The median price is 74.99 BRL, suggesting that half of the products are priced below this amount.

# Price distribution plot
plt.figure(figsize=(10, 6))
plt.hist(order_items['price'], bins=50, color='blue', edgecolor='black')
plt.title('Distribution of Product Prices')
plt.xlabel('Price (BRL)')
plt.ylabel('Frequency')
plt.savefig('../outputs/Documentation/images/price_distribution.png', 
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# There is a large variation in the prices. Let's see how the super cheap and super expensive products are distributed.

# Looking at the items under 5 BRL
cheap_items = order_items[order_items['price'] < 5]
print("\n**** Cheap Items (Price < 5 BRL) ****")
print(f"Items under 5 BRL: {len(cheap_items)}")
print(cheap_items['price'].describe())

# Looking at the items over 1000 BRL
expensive_items = order_items[order_items['price'] > 1000]
print("\n**** Expensive Items (Price > 1000 BRL) ****")
print(f"Items over 1000 BRL: {len(expensive_items)}")
print(expensive_items['price'].describe())


# There are 117 items that cost less than 5 BRL and 844 items that cost more than 1000 BRL. 
# The cheap items have a mean price of 3.61 BRL, while the expensive items have a mean price of 1588 BRL.

# Looking at the frieght cost. Here we look at it to see any anomolies
print("\n**** Freight Value Statistics ****")
print(order_items['freight_value'].describe())

# Interesting observation: On average the frieght cost is 19.9 BRL but the maximum is 409.68 BRL. 
# Either there are some very heavy items or there are some data entry errors or the items need to travel longer distance or need to be delivered to remote areas. 
# Let's look at the distribution of frieght costs.
plt.figure(figsize=(10, 6))
plt.hist(order_items['freight_value'], bins=50, color='orange', edgecolor='black')
plt.title('Distribution of Freight Values')
plt.xlabel('Freight Value (BRL)')
plt.ylabel('Frequency')
plt.savefig('../outputs/Documentation/images/freight_distribution.png', 
            dpi=300, bbox_inches='tight')
plt.show()
# The distribution of freight values is highly skewed, with most orders having a freight value of less than 50 BRL.


In [ ]:
# Lets analyse the products dataset to see what kind of products are being sold and if there are any patterns in the product categories.
print("\n**** Products Sample ****")
print(products.head())
print("\n**** Products Info ****")
print(products.info())
print("\n**** Products Description ****")
print(products.describe())

# Checking for missing values in products dataset
print("\n**** Products Missing Values ****")
print(products.isnull().sum())
# Looking at the product categories
print("\n**** Product Categories ****")
print(products['product_category_name'].value_counts())

# Merging products with the category name translation to get the English names of the categories
products_english = products.merge(product_category_name_translation, on='product_category_name', how='left')
print("\n**** Products with English Category Names ****")
print(products_english['product_category_name_english'].value_counts())

# Top categories by sales
order_items_products = order_items.merge(products_english[['product_id', 'product_category_name_english']], on='product_id', how='left')

# Counting the top categories by sales volume
print("\n**** Top Categories by Sales Volume ****")
top_categories = order_items_products['product_category_name_english'].value_counts().head(20)
print(top_categories)

# Top categories by total revenue
print("\n**** Top Categories by Revenue ****")
category_revenue = order_items_products.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(20)
print(category_revenue)


# Average price by category
print("\n**** Average Price by Category ****")
average_price_cat = order_items_products.groupby('product_category_name_english')['price'].mean().sort_values(ascending=False).head(20)
print(average_price_cat)


# Looking at the outputs from above, top 5 categories by sales volume are: bed_bath_table, health_beauty, sports_leisure, furniture_decor, computers_accessories.
# Top 5 categories by revenue are: health_beauty, watches_gifts, bed_bath_table, sports_leisure, computers_accessories.
# Looks like Bed Bath Table is the most popular category by sales volume, while Health Beauty generates the most revenue. 
# The average price of Bed Bath Table products is 93 BRL. But this category has the highest orders. Meaning, possibility of low priced items being sold in high volumes.
# Health beauty has high revenue but the volume is lower than the bed bath table category. Sports leisure also has strong revenue and string sales. 
# These three categories likely have high elasticity of demand.
# High price lower volume categories like computers, small appliances, and home appliances likely have low elasticity of demand. These are high price and low volume categories.
# Health Beauty - high revenue, high volume, room for optimization. Bed Bath Table - high volume, price sensitivity analysis. 
# Watches and gifts - high revenue, good everage price - sweet spot and potential for growth.

In [ ]:
## From the above I have observed some product categories fall under similar broad buckets. For example bed_bath_table and housewares fall under home essentials. 
## Similarly health_beauty and perfumes fall under personal care. 
# I will create some buckets and assign the products into these buckets. This way I can do some analysis at the bucket level and see if there are any patterns in the buckets. 
# This will also help me to identify the buckets that are performing well and the buckets that are not performing well. 
# Will be essential for pricing strategy and inventory management.

# Creating buckets for product categories
bucket_mapping = {
    'HOME_ESSENTIALS': ['bed_bath_table', 'housewares', 'furniture_decor', 'garden_tools', 
                        'home_appliances', 'kitchen_dining_laundry_garden_furniture', 
                        'home_construction', 'furniture_living_room', 'furniture_bedroom', 
                        'air_conditioning', 'home_confort', 'home_comfort_2', 
                        'furniture_mattress_and_upholstery'],
    'PERSONAL_CARE': ['health_beauty', 'perfumery', 'baby', 'diapers_and_hygiene'],
    'ELECTRONICS_TECH': ['computers_accessories', 'telephony', 'electronics', 'computers', 
                         'consoles_games', 'audio', 'tablets_printing_image', 'fixed_telephony',
                         'cine_photo'],
    'LEISURE_LIFESTYLE': ['sports_leisure', 'toys', 'watches_gifts', 'cool_stuff', 
                          'books_general_interest', 'musical_instruments', 'pet_shop', 
                          'fashion_bags_accessories', 'books_technical', 'books_imported',
                          'art', 'party_supplies', 'flowers', 'dvds_blu_ray', 'music',
                          'cds_dvds_musicals'],
    'AUTO_TOOLS': ['auto', 'construction_tools_construction', 'construction_tools_safety',
                   'costruction_tools_garden', 'construction_tools_lights', 
                   'costruction_tools_tools', 'signaling_and_security',
                   'agro_industry_and_commerce', 'industry_commerce_and_business'],
    'OFFICE_STATIONERY': ['stationery', 'office_furniture'],
    'FASHION_APPAREL': ['fashion_bags_accessories', 'fashion_shoes', 'fashion_male_clothing',
                        'fashion_underwear_beach', 'fashio_female_clothing', 'fashion_sport',
                        'fashion_childrens_clothes', 'luggage_accessories'],
    'FOOD_BEVERAGE': ['food', 'drinks', 'food_drink'],
    'SMALL_APPLIANCES': ['small_appliances', 'small_appliances_home_oven_and_coffee', 
                         'home_appliances_2'],
    'MISC': ['market_place', 'christmas_supplies', 'la_cuisine', 'security_and_services',
             'arts_and_craftmanship']
}

# Createing a reverse mapping to assign buckets to products
category_to_bucket = {}
for bucket, categories in bucket_mapping.items():
    for category in categories:
        category_to_bucket[category] = bucket

# Adding bucket column to products_english dataframe
products_english['bucket'] = products_english['product_category_name_english'].map(category_to_bucket)

# Adding bucket to order_items_products dataframe
order_items_with_bucket = order_items_products.merge(products_english[['product_id','bucket']], on='product_id', how='left')

# Checking the bucket distribution
print("\n**** Bucket Distribution (by order count) ****")
print(order_items_with_bucket['bucket'].value_counts())

print("\n**** Bucket Distribution (by revenue) ****")
bucket_revenue = order_items_with_bucket.groupby('bucket')['price'].sum().sort_values(ascending=False)
print(bucket_revenue)

print("\n**** Bucket Distribution (by average price) ****")
bucket_avg_price = order_items_with_bucket.groupby('bucket')['price'].mean().sort_values(ascending=False)
print(bucket_avg_price)

## Bucketing reveals more patterns
# Volume and revenue leaders: Leisure_Lifestyle - 2nd volume and 1 in revenue. Ave price is 145 BRL (Sweet Spot, high volume and decent prices), 
# Home_Essentials - 1 in volume, 2 in revenue, Avg price is 98 BRL (mass market, highest traffic, lower prices).
# Premium buckets: Small_applicances - Lowest volume, Highest avg price (likely low elasticity also functional purchases), 
# Auto_Tools - moderate volume, high avg prices (search-driven, comparison shopping). 

## Strategic opportunities: 
# Leisure_Lifestyle - 26.5K orders (24% of total), 3.8M BRL revenue (29% of total), 145 BRL avg.
#                     Highest revenue, strong volume, discretionary spending, potential for growth with targeted marketing and product expansion. Likely elastic.
# Home_Essentials - 33.8K orders (31% of total), 3.3M BRL revenue (25% of total), 98 BRL avg.
#                   High volume, competitive market, price-sensitive and potential for growth with competitive pricing and promotions.
# Personal_care - 16.2K orders (15% of total), 2.1M BRL revenue (16% of total), 128 BRL avg
#                 Strong revenue, moderate prices, brand loyealty, potential for growth with new product launches and marketing. Also likely inelastic.

## HYPOTHESES TO TEST IN ELASTICITY ANALYSIS:
# H1: Leisure/Lifestyle has higher price elasticity than Home Essentials (discretionary vs essential)
# H2: Personal Care shows brand loyalty (lower cross-elasticity within bucket)
# H3: Small Appliances are inelastic (functional, high-involvement purchases)
# H4: Home Essentials has strong cross-bucket substitution (budget constraints)

In [ ]:
## Lets look at price variation for each product
# This will reveal if there are products that are sold at very different price points, which could indicate different versions of the same product (e.g. basic vs premium) 
# or it could indicate price discrimination based on customer segments or price variation by seller or competition between sellers.

# Note: products with more variation is better for elasticity analysis as we can see how demand changes with price changes. 
# We will do similar analysis for the buckets as well to see if there are buckets that have more price variation than others. 
# This will help us to identify the buckets that are more likely to be elastic or inelastic.

price_variation = order_items_with_bucket.groupby('product_id').agg({
    'price': ['count', 'mean', 'std', 'min', 'max'],
    'product_category_name_english': 'first',
    'bucket': 'first'
}).reset_index()

price_variation.columns = ['product_id', 'order_count', 'price_mean', 'price_std', 'price_min', 'price_max', 'category', 'bucket']

# Calculating the coefficient of variation. This is a measure of relative variability and is calculated as the standard deviation divided by the mean.
price_variation['price_cv'] = price_variation['price_std'] / price_variation['price_mean']

# Lets filter out products with enough orders to have meaningful price variation analysis. We will look at products with at least 10 orders.
price_variation_volume = price_variation[price_variation['order_count'] >= 10]
print(f"Products with at least 10 orders: {len(price_variation_volume)}")

# Sorting the products by price variation (CV) to see which products have the most price variation and which have the least. 
# This will help us to identify products that are likely to be more elastic (high variation) and products that are likely to be more inelastic (low variation).
print("\n**** Products by high Price Variation (CV) ****")
print(price_variation_volume.nlargest(20, 'price_cv')[['product_id', 'category', 'bucket', 'order_count', 'price_mean', 'price_std', 'price_cv']])

print("\n**** Products by low Price Variation (CV) ****")
print(price_variation_volume.nsmallest(20, 'price_cv')[['product_id', 'category', 'bucket', 'order_count', 'price_mean', 'price_std', 'price_cv']])

## From the above outputs I noticed that different price variation exists for similar category. 
# Potential reason: I see different id for same category and bucket. This could be because of different sellers selling the same product at different prices, 
# or it could be because of different versions of the same product (e.g. basic vs premium) or different products under same category (Face cream vs face wash).
# Since we don't have the product characterestics data such as face cream vs face wash, 
# we will be looking at the price variation and seller variation in our analysis.

In [ ]:
## Correlation Analysis

# Price vs Freight correlation
print("\n**** Price vs Freight Correlation ****")
print(order_items_with_bucket[['price', 'freight_value']].corr())

# Scatter plot of price vs freight value
plt.figure(figsize=(10, 6))
plt.scatter(order_items_with_bucket['price'], order_items_with_bucket['freight_value'], alpha=0.1)
plt.title('Price vs Freight Cost')
plt.xlabel('Price (BRL)')
plt.ylabel('Freight Value (BRL)')
plt.xlim(0, 500) # Focusing on reasonable price range for better visualization
plt.ylim(0, 100) # Focusing on reasonable freight range for better visualization  
plt.savefig('../outputs/Documentation/images/price_freight_scatter.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# By bucket
print("\n**** Price vs Freight Correlation by Bucket ****")
for bucket in order_items_with_bucket['bucket'].dropna().unique():
    bucket_data = order_items_with_bucket[order_items_with_bucket['bucket'] == bucket]
    correlation = bucket_data[['price', 'freight_value']].corr().iloc[0, 1]
    print(f"{bucket}: {correlation:.2f}")


# The findings suggest that the price and frieght have moderate positive correlation overall 0.41. 
# Frieght likely driven by weight, distaince, and items characteristics.
# Looking at the bucket patterns: 
# Small Appliances have 0.57 correlation. Makes sense as the expensive appliances are heavy. 
# Rest of the buckets have moderate correlations. 
# This tells that the frieght is not a function of price. If it were, we would see high correlation value. 
# Category is is likely a driver - Small appliances are heavier and high price so high correlation. Food and Beverage - price not equal to weight so low correlation.
# The plot tells us that there are clear bands: 10-20 BRL, 30-40 BRL, 50-60 BRL, some outliers 80-100 BRL Frieght
# Suggests: Frieght is often standerdized, Not purely weight or price based, likely influenced by shipping zones.
# There is a weak linear relationship between price and frieght. Scattered vertically all over the price ranges. Confirming Freight is not a simple/linear function of price.
# Lots of noise in the relationship.
# Tells me that the frieght should be analysed seperately from price. Seller location matters. Customers likely see total cost (price + freight).

In [ ]:
# Lets perform similar analysis on price vs weight/volume
# Merging with product dimensions
order_items_full = order_items_with_bucket.merge(products_english[['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']], 
                                                 on='product_id', how='left')

# Calculating the volume
order_items_full['product_volume_cm3'] = order_items_full['product_length_cm'] * order_items_full['product_height_cm'] * order_items_full['product_width_cm']

# Price vs Weight correlation
print("\n**** Price vs Weight Correlation ****")
correlations = order_items_full[['price', 'freight_value' ,'product_weight_g', 'product_volume_cm3']].corr()
print(correlations)

# Plotting the correlations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Price vs Weight
axes[0].scatter(order_items_full['product_weight_g'], order_items_full['price'], alpha=0.1)
axes[0].set_title('Price vs Weight')
axes[0].set_xlabel('Weight (g)')
axes[0].set_ylabel('Price (BRL)')
axes[0].set_xlim(0, 5000) # Focusing on reasonable weight range for better visualization
axes[0].set_ylim(0, 500) # Focusing on reasonable price range for better visualization

# Price vs Volume
axes[1].scatter(order_items_full['product_volume_cm3'], order_items_full['price'], alpha=0.1)
axes[1].set_title('Price vs Volume')
axes[1].set_xlabel('Volume (cm3)')
axes[1].set_ylabel('Price (BRL)')
axes[1].set_xlim(0, 5000000) # Focusing on reasonable volume range for better visualization
axes[1].set_ylim(0, 500) # Focusing on reasonable price range for better visualization

# Frieght vs Weight
axes[2].scatter(order_items_full['product_weight_g'], order_items_full['freight_value'], alpha=0.1)
axes[2].set_title('Freight Value vs Weight')
axes[2].set_xlabel('Weight (g)')
axes[2].set_ylabel('Freight Value (BRL)')
axes[2].set_xlim(0, 5000) # Focusing on reasonable weight range for better visualization
axes[2].set_ylim(0, 100) # Focusing on reasonable freight range for better visualization
plt.tight_layout()

plt.savefig('../outputs/Documentation/images/price_weight_volume.png', 
            dpi=300, bbox_inches='tight')

plt.show()

## Key findings:
# Price vs Weight: 0.34 weak correlation. Most products clusted at low weight. Price is not driven by weight. 
#   Insights: Price reflects value (brand, features, quality), not just weight. Light items can be expensive (e.g. jewelry) and heavy items can be cheap (e.g. books). 
#             Customers pay for what product does not the weight.
# Price vs Volume: 0.30 weak correlation. Similar to weight, price is not strongly driven by volume.
#   Insights: This confirms value based pricing not cost-based. Small products can be expensive (e.g. electronics) and large products can be cheap (e.g. furniture). 
#             Volume alone does not determine price.
# Freight vs Weight: 0.61 String correlation. The scatter plot shows a clear horizontal banding pattern. But frieght increases with weight. 
#                    Heavier items generally cost more to ship, but there is variability likely due to shipping zones and carrier pricing.
#   Insights: Freight follows shipping logic. But also shows tiered pricing. This is a logistics cost, not a customer cost. 
#             Customers likely see total cost (price + freight) when making purchase decisions.
# Overall, price is influenced by value-based factors rather than physical characteristics, while freight is more closely tied to weight with some variability.
# The big insight is that price is not determined by physical attributes, but by brand, features, quality, and perceived value.
# Freight is more of a function of Weight and Volume. This is a logistics reality.

# Price elsaticity should be estimated independently of freight: Customers see total cost. But the pricing decisions focus on value not shipping.
# High-value, low-weight products create a best margin opportunity. Electronics, Jewelry, cosmetics. These can be priced aggressively for growth. High price, low frieght.
# Bulky, low-price products are margin challenges. Furniture, home applicances. These require cost optimization and competitive pricing. Low price, high frieght.

In [ ]:
# Correlations by bucket
print("\n**** Correlations by Bucket ****")
for bucket in order_items_full['bucket'].dropna().unique():
    bucket_data = order_items_full[order_items_full['bucket'] == bucket]
    print(f"\nBucket: {bucket}")
    corr_matrix = bucket_data[['price', 'freight_value' ,'product_weight_g', 'product_volume_cm3']].corr()
    print(corr_matrix)

# Insights by bucket:
# Price driven by weight: Misc, Office_stationary, Small_appliances, Home_essentials.
# Price driven by value: Leisure_lifestyle, ELectronics_tech, Food_beverage.
# Home_Essentials: 0.48. Price is moderately driven by weight. Furniture, appliances are functional purchases. Here the pricing strategy would be cost plus likely materials and shipping. 
#                  For elasticity analysis this woulr be moderate elasticity.
# Personal_care: 0.43. Price somewhat tied to weight, larger bottles higher price. Also brand premium. Pricing strategy could be value based plus cost. 
#                Elasticity could be moderate, with some brand loyalty.
# Electronice_Tech: 0.37. Price is weakly correlated with weight. High value items like phones and laptops can be light but expensive. Pricing strategy is value based. 
#                   Elasticity could be moderate to high depending on the product. This is a research driven purchases.
# Leisure_Lifestyle: 0.31. Lowest physical correlation. Price driven by brand, trends, and discretionary nature. Pricing strategy is value based with room for promotions.
#                    Elasticity likely high due to discretionary spending. This sould be the best bucket for pricing optimization.
# Small_Appliances: 0.52 price weight and 0.57 price freight. Price and Freight are driven by weight. These are functional purchases. 
#                   Pricing strategy is likely cost plus with focus on materials and shipping. Elasticity likely low (functional, less substitutable).
# Food_Beverage: 0.44 Price weight and 0.28 price freight. Price is somewhat tied to weight but not freight. Likely standerdized packaging, local sourcing. 
#                Pricing strategy could be competitive (low margins). ELasticity could be moderate to high due to availability of substitutes and price sensitivity in food.

# Based on the above correlations:
# Most Elastic (Price Sensitive): Leisure_Lifestyle (Discretionary, value based, substitutable), Food_Beverage (Commodity like), ELectronics_Tech (Research driven, comparable)
# Least ELastic (Price Insensitive): Small_Appliances (Functional, cost driven), Home_Essentials (Large, Considered purchases), Office_Stationary (Business purchases, Specific needs)
 

In [ ]:
# Temporal patterns:

# Merging with orders data to get order date
orders_full = orders.merge(order_items_with_bucket, on='order_id', how='inner')


# Converting order_purchase_timestamp to datetime
orders_full['order_purchase_timestamp'] = pd.to_datetime(orders_full['order_purchase_timestamp'])
orders_full['day_of_week'] = orders_full['order_purchase_timestamp'].dt.day_name()
orders_full['month'] = orders_full['order_purchase_timestamp'].dt.month
orders_full['year_month'] = orders_full['order_purchase_timestamp'].dt.to_period('M')

# Overall volume and price trends over time
print("\n**** Overall Volume and Price Trends ****")
monthly_summary = orders_full.groupby('year_month').agg({
    'order_id': 'count',
    'price': 'mean'
}).reset_index()
monthly_summary.columns = ['year_month', 'order_count', 'avg_price']
print(monthly_summary)

# Plotting the trends
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Volume trend
axes[0].plot(monthly_summary['year_month'].astype(str), monthly_summary['order_count'])
axes[0].set_title('Monthly Order Volume')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Order Count')
axes[0].tick_params(axis='x', rotation=45)

# Price trend
axes[1].plot(monthly_summary['year_month'].astype(str), monthly_summary['avg_price'])
axes[1].set_title('Monthly Average Price')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Price (BRL)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/Documentation/images/temporal_volume_price.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# Day of the week patterns
print("\n**** Price by Day of the Week Patterns ****")
day_summary = orders_full.groupby('day_of_week')['price'].agg(['mean', 'count'])
print(day_summary)

# Month patterns
print("\n**** Price by Month Patterns ****")
month_summary = orders_full.groupby('month')['price'].agg(['mean', 'count'])
print(month_summary)


# Key findings:
# Volume Trends: 1. Sept 2016 ~0 orders. 2. Rapid growth through 2017. 3. Peak Nov 2017 - Black friday / holiday season. 4. Stable volume Jan - Aug 2018, 5. Sharp drop : Sep 2018 (data cutoff).
# Price Trends: 1. Oct 2016 136 BRL (small sample), 2. Dec 2016 11 BRL (only one order. ignore this). 3. Jan 2017 125 BRL (higher initial prices),
#               4. Feb - Aug 2017 Steady decline to ~110 - 120 BRL, 5. Sept 2017 - Aug 2018 Stable ~115-125 BRL
# No major seasonal price variation (this is good for analysis). Average price is relatively stable over time. 
# This means price changes are likely driven by product mix and category trends rather than seasonal promotions or discounts.
# Day of the week patterns: Price - 1. Saturday 123.60 BRL (highest, wekeend shopping, more leisure maybe), 2. Sunday 118.46 BRL (lowest), 3. Weekdays 119-122 BRL (stable)
#                           Volume - 1. Monday 18393 (most orders, start of the week shopping), 2. Tuesday 18237, 
#                                    3. Saturday 12168 (fewer orders despite higher prices)
# Day of the week is not a major pricing factor. Volume shifts but prices stay stable.
# Monthly Patterns: Highest Prices - 1. September 129 BRL (back to school? looks like limited data), 2. April 1278 BRL (Easter holiday?), 3. Oct 126 BRL (pre-holiday season)
#                   Lowest Prices - 1. February 113 BRL (post-holiday clearance?), 2. January 117 BRL (New Year Deals?), 3. August 118 BRL (Summer lull?)
# Minimal seasonal price variation, No aggressive promotional cycles, Marketplace pricing stays stable

# Critical for elasticity estimation:
# Stable pricing environment: No major seasonal discounts or promotions, which allows for cleaner elasticity estimation.
# Price changes I observe are seller or product decisions not seasonal.
# Elasticity estimation can focus on product and category level variations without needing to control heavily for time-based promotions.
# Cleaner causal identification: With stable prices over time, we can more confidently attribute changes in demand to price changes rather than seasonal factors.

# Data quality notes:
# September 2016 and September 2018 has very low order counts. This could be because of data cutoff or entry errors. 
# Strong period: Jan 2017 - Aug 2018 with stable volume and prices. This should be the focus period for elasticity analysis.


In [ ]:
# Repeat Purchase Analysis

# Couting orders by customer
repeat_purchases = customers.groupby('customer_unique_id').agg({
    'customer_id': 'count'
}).reset_index()

repeat_purchases.columns = ['customer_unique_id', 'num_orders']

# Summary statistics for repeat purchases
print("\n**** Repeat Purchase Distribution ****")
print(f"Total unique customers: {len(repeat_purchases)}")
print(f"Total orders: {repeat_purchases['num_orders'].sum()}")
print(f"Customers with 1 order: {(repeat_purchases['num_orders'] == 1).sum()}")
print(f"Customers with 2 or more orders: {(repeat_purchases['num_orders'] > 1).sum()}")
print(f"Repeat purchase rate: {(repeat_purchases['num_orders']>1).sum() / len(repeat_purchases) * 100:.2f}%")
# Distribution of order counts
print("\n Orders per customer distribution")
print(repeat_purchases['num_orders'].value_counts().sort_index().head(10))

# Plotting the distribution of orders per customer
plt.figure(figsize=(10, 6))
repeat_purchases['num_orders'].value_counts().sort_index().head(10).plot(kind='bar')
plt.title('Distribution of Orders per Customer')
plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')
plt.savefig('../outputs/Documentation/images/orders_per_customer.png', 
            dpi=300, bbox_inches='tight')
plt.show()

## Key Insight: Very low repeat purchase rate of 3.12%. 
# 96096 unique customers made 99411 orders. 97% of them are one-time buyers. Only 3% made 2 or more purchases. This indicates this is not a loyelty driven marketplace.
## Implications:
# Customer acquisition Market is the key to growth:
#       Customers come, buy once, and leave. This means the focus should be on acquiring new customers rather than relying on repeat purchases.
#       There is a very little brand/seller loyalty.
#       Price shopping is likely common. Customers may be looking for the best deal for each purchase rather than sticking to a particular seller or brand.
#       For Pricing strategy: Can't relay on customer loyalty. Every transaction is likely a first-time buyer. 
#                             Price elasticity is likely high as customers are price shopping and not loyal.
#                             Competitive pricing is critical.

# Why so low repeat purchase rate?
# 1. Platform aggregation effect: 
#    Customers are loyal to platforms not sellers. 
#    Next purchase might be from a different seller or non-olist seller. 
#    Platform shows the "best deal". So customers can easily switch the sellers. No switching cost.
# 2. Product mix effect:
#    Many categories are one-time purchases (e.g. furniture, appliances).
#    Few consumables (we saw only 1167 food/beverage orders).
#    People don't buy beds or TVs frequently. This naturally limits repeat purchases.
# 3. Brazilian e-commerce dynamics:
#    High marketplace fragmentation. Many sellers competing for price-sensitive customers.
#    Price comparison culture. Customers actively seek the best deal for each purchase.
#    Low brand loyalty in online shopping.

# For customers who do repeat purchase, let's look at their behavior:
# Do they buy from the same category or different?
# Do they pay higher/lower prices on repeat?
# Which buckets have highest repeat purchase rates?
# Are they more or less price sensitive?

In [ ]:
# Repeat purchaase analysis of repeat buyers:

# For the 2997 repeat customers, which buckets do they buy from?
repeat_customer_ids = repeat_purchases[repeat_purchases['num_orders'] > 1]['customer_unique_id']

repeat_customer_orders = customers[customers['customer_unique_id'].isin(repeat_customer_ids)]
repeat_orders_full = repeat_customer_orders.merge(orders, on = 'customer_id').merge(order_items_with_bucket, on = 'order_id')

print("\n**** Repeat Purchases by Bucket ****")
repeat_by_bucket = repeat_orders_full.groupby('bucket').agg({
    'customer_unique_id': 'nunique',
    'order_id': 'nunique'
}).sort_values(by='customer_unique_id', ascending=False)
repeat_by_bucket.columns = ['unique_repeat_customers', 'total_orders']
print(repeat_by_bucket)

# Repeat rate by bucket
total_customers_by_bucket = orders.merge(customers, on = 'customer_id').merge(order_items_with_bucket, on = 'order_id').groupby('bucket')['customer_unique_id'].nunique()

repeat_by_bucket['total_customers'] = total_customers_by_bucket
repeat_by_bucket['repeat_rate'] = repeat_by_bucket['unique_repeat_customers'] / repeat_by_bucket['total_customers'] * 100
print("\n**** Repeat Purchase Rate by Bucket ****")
print(repeat_by_bucket[['repeat_rate']].sort_values(by='repeat_rate', ascending=False))

## Key Insights:
# Fashion and Food have highest repeat purchase rates. 
# Possible explanation on fashion: 1. Consumable in nature: Clothes wear out, Fashion trends change, Accessories are impulse purchases, 
# 2. Low price points: Easier to make repeat purchases, less financial commitment, more experimentation.
# 3. Gift purchases: Customers may buy multiple gifts over time, especially around holidays.
# Why home essentials is high in unique repeat customers: 
# 1. Multi room or multi-item needs: Buys kitchen items, then bathroom. Home improvement projects often require multiple purchases over time.
# 2. Complementary purchases: Buy a dining table, then later buy chairs. Initial purchase leads to follow-up purchases.
# 3. Household growth or turnovers: New homeowners or growing families may make multiple purchases as they furnish their homes. 

# Pricing Strategy Implications:
# 1. Fashion Apparel: 
#   High repeat rate suggests opportunity for loyalty programs. First purchase can be a gateway to future purchases (competitive pricing, personalized recommendations).
#   Once the customer buys a fashion item, they are more likely to come back for more. 
#   This means we can afford to be more aggressive on pricing for the first purchase to acquire the customer, knowing that there is a good chance of repeat business.
# 2. Food and Beverage:
#   High repeat rate indicates potential for subscription models or bundle offers. 
#   Customers may be looking for convenience and value. Offering discounts on repeat purchases or creating bundles can encourage continued buying.
# 3. Home Essentials:
#   Moderate repeat rate suggests focus on cross-selling and upselling.
#   If a customer bought bed, they might need bedding, decor, or furniture. So recommed these additional products to increase the lifetime value of the customer.
#   Sequential purchasing behavior (buying related items over time) can be leveraged with targeted marketing and personalized recommendations.
# 4. ELectronics: Low repeat rate suggests focus on acquisition and competitive pricing.
#    Electronics are often one-time purchases with long replacement cycles. 
#    The strategy should be to attract new customers with competitive pricing and promotions, rather than relying on repeat business.

In [ ]:
## Category Loyalty Analysis:
print("\n**** Category Loyalty Analysis ****")

# For the 2997 repeat customers, how many buckets do they shop from?
customer_bucket_diversity = repeat_orders_full.groupby('customer_unique_id').agg({
    'bucket': lambda x: x.nunique(),
    'product_category_name_english': lambda x: x.nunique(),
    'order_id': 'nunique'
}).reset_index()

customer_bucket_diversity.columns = ['customer_unique_id', 'num_buckets', 'num_categories', 'num_orders']

print("\n**** Bucket Loyalty (2,997 repeat customers) ****")
print(f"Customers who bought from 1 bucket only: {(customer_bucket_diversity['num_buckets'] == 1).sum()} ({(customer_bucket_diversity['num_buckets'] == 1).sum()/len(customer_bucket_diversity)*100:.1f}%)")
print(f"Customers who bought from 2 buckets: {(customer_bucket_diversity['num_buckets'] == 2).sum()} ({(customer_bucket_diversity['num_buckets'] == 2).sum()/len(customer_bucket_diversity)*100:.1f}%)")
print(f"Customers who bought from 3+ buckets: {(customer_bucket_diversity['num_buckets'] >= 3).sum()} ({(customer_bucket_diversity['num_buckets'] >= 3).sum()/len(customer_bucket_diversity)*100:.1f}%)")

print(f"\n**** Category Diversity ****")
print(f"Average buckets per repeat customer: {customer_bucket_diversity['num_buckets'].mean():.2f}")
print(f"Average categories per repeat customer: {customer_bucket_diversity['num_categories'].mean():.2f}")

# Distribution
print("\n**** Distribution of Buckets per Customer ****")
print(customer_bucket_diversity['num_buckets'].value_counts().sort_index())

# Plotting the distribution of buckets per customer
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Buckets per customer
customer_bucket_diversity['num_buckets'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Number of Buckets per Repeat Customer')
axes[0].set_xlabel('Number of Different Buckets')
axes[0].set_ylabel('Number of Customers')

# Categories per customer
customer_bucket_diversity['num_categories'].value_counts().sort_index().head(15).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Number of Categories per Repeat Customer')
axes[1].set_xlabel('Number of Different Categories')
axes[1].set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('../outputs/Documentation/images/bucket_loyalty.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# Key Insights:
# 60% of the repeat customers stick to one bucket. 37.9% shop from 2 buckets, and 1.3% shop from 3 or more buckets.
# This indicates a moderate level of category loyalty among repeat customers. They find what they like and stick with it


In [ ]:
## Price Comparison: First-time vs Repeat Customers:

print("\n**** Price Comparison: First-time vs Repeat Customers ****")

# Tag orders as first-time or repeat
orders_tagged = customers.merge(orders, on='customer_id').sort_values('order_purchase_timestamp')
orders_tagged['order_number'] = orders_tagged.groupby('customer_unique_id').cumcount() + 1
orders_tagged['customer_type'] = orders_tagged['order_number'].apply(
    lambda x: 'First-time' if x == 1 else 'Repeat'
)

# Merge with order items to get price and bucket information
orders_with_prices = orders_tagged.merge(order_items_with_bucket, on='order_id')

# Overall price comparison
print("\n**** Overall Price Comparison ****")
overall_price_comp = orders_with_prices.groupby('customer_type')['price'].agg(['mean', 'median', 'std', 'count']).round(2)
print(overall_price_comp)

# Statistical test for price difference
from scipy import stats
first_time_prices = orders_with_prices[orders_with_prices['customer_type'] == 'First-time']['price']
repeat_prices = orders_with_prices[orders_with_prices['customer_type'] == 'Repeat']['price']
t_stat, p_value = stats.ttest_ind(first_time_prices, repeat_prices)
print(f"\nT-test for Price Difference: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

if p_value < 0.05:
    print("The difference in average price between first-time and repeat customers is statistically significant.")
else:
    print("The difference in average price between first-time and repeat customers is not statistically significant.")

# By bucket price comparison
print("\n**** Price Comparison by Bucket ****")
bucket_price_comp = orders_with_prices.groupby(['bucket', 'customer_type'])['price'].agg(['mean', 'median', 'std', 'count']).round(2)
print(bucket_price_comp)

# Calculate repeat premium/discount by bucket
price_pivot = orders_with_prices.pivot_table(
    values = 'price',
    index = 'bucket',
    columns = 'customer_type',
    aggfunc = 'mean'
)
price_pivot['repeat_premium_pct'] = ((price_pivot['Repeat'] - price_pivot['First-time']) / price_pivot['First-time']) * 100

price_pivot['price_diff_BRL'] = price_pivot['Repeat'] - price_pivot['First-time']

print("\n**** Repeat Customer Price Premium/Discount ****")
print(price_pivot[['First-time', 'Repeat', 'price_diff_BRL', 'repeat_premium_pct']].sort_values('repeat_premium_pct', ascending=False).round(2))

#Plotting the price comparison by bucket
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall distribution
orders_with_prices.boxplot(column='price', by='customer_type', ax=axes[0])
axes[0].set_title('Price Distribution: First-time vs Repeat')
axes[0].set_xlabel('Customer Type')
axes[0].set_ylabel('Price (BRL)')
axes[0].set_ylim(0, 300)
plt.sca(axes[0])
plt.xticks(rotation=0)

# By bucket
top_buckets = ['HOME_ESSENTIALS', 'LEISURE_LIFESTYLE', 'PERSONAL_CARE', 'ELECTRONICS_TECH', 'FASHION_APPAREL']
bucket_data = orders_with_prices[orders_with_prices['bucket'].isin(top_buckets)]
bucket_price_pivot = bucket_data.pivot_table(values='price', index='bucket', columns='customer_type', aggfunc='mean')
bucket_price_pivot.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Average Price by Bucket: First-time vs Repeat')
axes[1].set_xlabel('Bucket')
axes[1].set_ylabel('Average Price (BRL)')
axes[1].legend(title='Customer Type')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/Documentation/images/price_comparison.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# Key Insights:
# Repeat customers pay significantly less: Overall the first-time customers pay on average 121.31 and repeat customers pay 102.74. So the discount is -18.57 BRL (-15.3%).
# This is statistically significant with p-value < 0.05. This suggests that repeat customers are more price sensitive or are buying lower-priced items.
# However, is conclusion may be confounded by the category inside each bucket. 
# For example, in Home essentials, repeat customers may be buying more low-priced decor items while first-time buyers are buying higher-priced furniture.
# Hence we must look at the price comparison by bucket to get a clearer picture.

In [ ]:
# Same SKU repeat purchase analysis:
print("\n**** Same SKU Repeat Purchase Analysis ****")

# Find customers who bought exact same product_id 2+ times
same_sku_purchases = repeat_orders_full.groupby(['customer_unique_id', 'product_id']).agg({
    'price': ['first', 'last', 'count'],
    'order_purchase_timestamp': ['first', 'last'],
    'product_category_name_english': 'first',
    'bucket': 'first'
}).reset_index()

same_sku_purchases.columns = ['customer_unique_id', 'product_id', 'first_price', 'last_price', 
                               'num_purchases', 'first_date', 'last_date', 'category', 'bucket']


# Filter to 2+ purchases of same SKU
same_sku_repeat = same_sku_purchases[same_sku_purchases['num_purchases'] >= 2]

print(f"\n**** Same SKU Repurchases ****")
print(f"Customers who repurchased exact same product: {same_sku_repeat['customer_unique_id'].nunique()}")
print(f"Total same-SKU repurchase instances: {len(same_sku_repeat)}")
print(f"Percentage of repeat customers: {same_sku_repeat['customer_unique_id'].nunique() / 2997 * 100:.1f}%")

# Price change analysis for same SKU repurchases
same_sku_repeat['price_change_BRL'] = same_sku_repeat['last_price'] - same_sku_repeat['first_price']
same_sku_repeat['price_change_pct'] = (same_sku_repeat['price_change_BRL'] / same_sku_repeat['first_price']) * 100

print("\n**** Price Change for Same SKU Repurchases ****")

print(f"Average: {same_sku_repeat['price_change_BRL'].mean():.2f} BRL ({same_sku_repeat['price_change_pct'].mean():.1f}%)")
print(f"Median: {same_sku_repeat['price_change_BRL'].median():.2f} BRL ({same_sku_repeat['price_change_pct'].median():.1f}%)")

print(f"\nPaid more on repeat: {(same_sku_repeat['price_change_BRL'] > 0).sum()} ({(same_sku_repeat['price_change_BRL'] > 0).sum()/len(same_sku_repeat)*100:.1f}%)")
print(f"Paid same: {(same_sku_repeat['price_change_BRL'] == 0).sum()} ({(same_sku_repeat['price_change_BRL'] == 0).sum()/len(same_sku_repeat)*100:.1f}%)")
print(f"Paid less on repeat: {(same_sku_repeat['price_change_BRL'] < 0).sum()} ({(same_sku_repeat['price_change_BRL'] < 0).sum()/len(same_sku_repeat)*100:.1f}%)")

# By bucket
print("\n**** Same sku Price Change by Bucket ****")
bucket_same_sku = same_sku_repeat.groupby('bucket').agg({
    'price_change_pct': ['mean', 'median', 'count'],
    'customer_unique_id': 'nunique'
}).round(2)
print(bucket_same_sku)

# By category (top categories only)
print("\n**** Same sku Repurchase by Category (Top 10) ****")
category_same_sku = same_sku_repeat.groupby('category').size().sort_values(ascending=False).head(10)
print(category_same_sku)

# Key Insights:
# From the analysis on same SKU repurchases or different shed light on the repeat customers behavior.
# We find that only 26% of the repeat customers purchase same product/SKU. When they do repurchase same SKU 90.3% pay exactly the same price. 
# only 5.9% pay less and only 3.9% pay more. 
# Average price change is *0.2% or -0.71 BRL. Essentially zero. This indicates the customers are not bargain hunting for the same product. They are likely loyal to the product and willing to pay the same price again.

In [ ]:
# Product mix shift analysis for repeat customers: Same category different SKU analysis

print("\n**** Product mix shift analysis ****")

# For each repeat customer in a category, track product diversity
customer_category_products = repeat_orders_full.groupby(['customer_unique_id', 'product_category_name_english']).agg({
    'product_id': lambda x: list(x.unique()),
    'price': ['first', 'last', 'mean'],
    'order_id': 'count'
}).reset_index()

customer_category_products.columns = ['customer_unique_id', 'category', 'product_ids', 
                                       'first_price', 'last_price', 'avg_price', 'num_orders']

# Add diversity metric
customer_category_products['num_different_products'] = customer_category_products['product_ids'].apply(len)
customer_category_products['price_change_BRL'] = customer_category_products['last_price'] - customer_category_products['first_price']
customer_category_products['price_change_pct'] = (customer_category_products['price_change_BRL'] / customer_category_products['first_price']) * 100

# Filter to customers who bought from same category 2+ times
multi_order_same_cat = customer_category_products[customer_category_products['num_orders'] >= 2]

print(f"\n**** Customers with 2+ orders in same category ****")
print(f"Total customer-category combinations: {len(multi_order_same_cat)}")
print(f"Bought same product only: {(multi_order_same_cat['num_different_products'] == 1).sum()} ({(multi_order_same_cat['num_different_products'] == 1).sum()/len(multi_order_same_cat)*100:.1f}%)")
print(f"Bought different products: {(multi_order_same_cat['num_different_products'] > 1).sum()} ({(multi_order_same_cat['num_different_products'] > 1).sum()/len(multi_order_same_cat)*100:.1f}%)")

# For those who switched products, what was price impact?
product_switchers = multi_order_same_cat[multi_order_same_cat['num_different_products'] > 1]

print(f"\n**** Product Switchers (bought different SKUs within same category) ****")
print(f"Number of switchers: {len(product_switchers)}")
print(f"Average price change: {product_switchers['price_change_BRL'].mean():.2f} BRL ({product_switchers['price_change_pct'].mean():.1f}%)")
print(f"Median price change: {product_switchers['price_change_BRL'].median():.2f} BRL ({product_switchers['price_change_pct'].median():.1f}%)")

print(f"\nPaid less after switching: {(product_switchers['price_change_BRL'] < 0).sum()} ({(product_switchers['price_change_BRL'] < 0).sum()/len(product_switchers)*100:.1f}%)")
print(f"Paid more after switching: {(product_switchers['price_change_BRL'] > 0).sum()} ({(product_switchers['price_change_BRL'] > 0).sum()/len(product_switchers)*100:.1f}%)")
print(f"Paid same: {(product_switchers['price_change_BRL'] == 0).sum()}")

# Average products purchased per category
print(f"\n**** Average Different Products per Category ****")
print(f"Mean: {multi_order_same_cat['num_different_products'].mean():.2f}")
print(f"Median: {multi_order_same_cat['num_different_products'].median():.0f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of product diversity
multi_order_same_cat['num_different_products'].value_counts().sort_index().head(10).plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('Product Diversity: Repeat Customers in Same Category')
axes[0].set_xlabel('Number of Different Products Purchased')
axes[0].set_ylabel('Number of Customer-Category Instances')

# Price change distribution for switchers
product_switchers['price_change_pct'].hist(bins=30, ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Price Change for Product Switchers')
axes[1].set_xlabel('Price Change (%)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2)
plt.tight_layout()
plt.savefig('../outputs/Documentation/images/product_diversity.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# Key Insights:
# 67% of the repeat customers bought different products. 
# 1186 customers switched while 588 bought the same product. 
# When Switching products, 41.4% pay less and 42.8% pay more. Average price change is +4.19 BRL. 
# This indicates that for customers who buy within the same category, there is a lot of product exploration. They are not loyal to a specific SKU but rather to the category.
# Customers are buying different often cheaper products. For instance in home essentials, they might buy a bed frame first and then bedding. 
# Product mix is the primary driver of the repeat purchases. 
# Seconday driver is the same product repurchase with no price chanage. Suitable for consumables or products with high satisfaction.
# For pricing strategy, this means that for repeat customers, we should focus on category-level promotions and recommendations rather than SKU-level.
# Pricing stratgegy implications: Repeat customers explore product portfolios, buying cheaper complementary items. 
# Focus on cross-selling and category-level promotions to maximize lifetime value. Not on discounts. Loyalty is more to the category than the specific product. 

# The visualization tells us that most customers buy 2 different products within the same category. Some buy same product. There is a clear product exploration pattern.
# When it comes to price change it remains to be no change with some instances of price decreases. Most of the consumers see no price change and some trading down.
# Our original fininding of price differences between first-time and repeat customers is likely driven by the product mix shift rather than price sensitivity.
# The -15.3% discount is from the natural category exploration, not price sensitivity. Focus on cross-sell, not dicsounts. 

In [ ]:
from PIL import Image
import os

os.chdir('../outputs/Documentation/images')

for filename in os.listdir('.'):
    if filename.endswith('.png'):
        img = Image.open(filename)
        
        # Resize to max width 1200px
        if img.width > 1200:
            ratio = 1200 / img.width
            new_height = int(img.height * ratio)
            img = img.resize((1200, new_height), Image.Resampling.LANCZOS)
        
        # Save with compression
        img.save(filename, optimize=True, quality=60)
        
        size_kb = os.path.getsize(filename) / 1024
        print(f"{filename}: {size_kb:.1f} KB")

In [ ]:
# Check payment method distribution
print(payments['payment_type'].value_counts())

# Check installment patterns
print(payments['payment_installments'].value_counts().head(10))

# Orders with multiple payments
multi_payment_orders = payments.groupby('order_id').size()
print(f"Orders with 1 payment: {(multi_payment_orders == 1).sum()}")
print(f"Orders with 2+ payments: {(multi_payment_orders > 1).sum()}")

In [ ]:
# For distance analysis
# Calculate customer-seller distance (requires geolocation merge)
# Could be used as freight cost control variable